# # 介紹
本程式使用 pyautogui 來完成一個 Blitz 的遊戲機器人。

In [24]:
import pyautogui as pyg
pyg.sleep (2)
basex=514
basey=391
gap=40

In [25]:
for row in range(8):
    for col in range(8):
        x = basex + col * gap
        y = basey + row * gap
        color = pyg.pixel(x, y)
        print(row, col, x, y, color)

0 0 514 391 (25, 26, 27)
0 1 554 391 (25, 26, 27)
0 2 594 391 (25, 26, 27)
0 3 634 391 (25, 26, 27)
0 4 674 391 (25, 26, 27)
0 5 714 391 (25, 26, 27)
0 6 754 391 (18, 19, 20)
0 7 794 391 (18, 19, 20)
1 0 514 431 (25, 26, 27)
1 1 554 431 (25, 26, 27)
1 2 594 431 (25, 26, 27)
1 3 634 431 (25, 26, 27)
1 4 674 431 (25, 26, 27)
1 5 714 431 (25, 26, 27)
1 6 754 431 (18, 19, 20)
1 7 794 431 (18, 19, 20)
2 0 514 471 (25, 26, 27)
2 1 554 471 (25, 26, 27)
2 2 594 471 (25, 26, 27)
2 3 634 471 (25, 26, 27)
2 4 674 471 (25, 26, 27)
2 5 714 471 (25, 26, 27)
2 6 754 471 (18, 19, 20)
2 7 794 471 (18, 19, 20)
3 0 514 511 (25, 26, 27)
3 1 554 511 (25, 26, 27)
3 2 594 511 (25, 26, 27)
3 3 634 511 (25, 26, 27)
3 4 674 511 (25, 26, 27)
3 5 714 511 (25, 26, 27)
3 6 754 511 (18, 19, 20)
3 7 794 511 (18, 19, 20)
4 0 514 551 (25, 26, 27)
4 1 554 551 (25, 26, 27)
4 2 594 551 (25, 26, 27)
4 3 634 551 (25, 26, 27)
4 4 674 551 (25, 26, 27)
4 5 714 551 (25, 26, 27)
4 6 754 551 (18, 19, 20)
4 7 794 551 (18, 19, 20)


In [26]:
def gem_position(r, c):
    x = basex + c * gap
    y = basey + r * gap
    return (x, y)

In [27]:
print(gem_position(0,0))
print(gem_position(0,1))
print(gem_position(1,0))
      

(514, 391)
(554, 391)
(514, 431)


In [28]:
def get_gem_color(r, c, im):
    x, y = gem_position(r, c)
    return im.getpixel((x, y))

In [29]:
im = pyg.screenshot()
for i in range(8):
    x,y = gem_position(0,i)
    pyg.click(x,y, duration=1)
    print(get_gem_color(0,i,im))

(25, 26, 27)
(25, 26, 27)
(25, 26, 27)
(25, 26, 27)
(25, 26, 27)
(25, 26, 27)
(18, 19, 20)
(18, 19, 20)


In [30]:
def can_move_left(r, c, im):
    if not (0 <= r < 8 and 1 <= c < 8):
        return False

    current_color = get_gem_color(r, c, im)
    left_color = get_gem_color(r, c - 1, im)

    def color_after_swap(row, col):
        if (row, col) == (r, c):
            return left_color
        if (row, col) == (r, c - 1):
            return current_color
        return get_gem_color(row, col, im)

    def has_match_at(row, col):
        color = color_after_swap(row, col)
        for dr, dc in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                next_row = row + dr * direction
                next_col = col + dc * direction
                while (0 <= next_row < 8 and 0 <= next_col < 8
                       and color_after_swap(next_row, next_col) == color):
                    count += 1
                    next_row += dr * direction
                    next_col += dc * direction
            if count >= 3:
                return True
        return False

    return has_match_at(r, c - 1) or has_match_at(r, c)

In [35]:
im=pyg.screenshot()
print(can_move_left(7,5,im))
print(can_move_left(0,0,im))
print(can_move_left(1,1,im))

True
False
True


In [ ]:
def _can_swap_and_match(r, c, other_r, other_c, im):
    if not (0 <= r < 8 and 0 <= c < 8
            and 0 <= other_r < 8 and 0 <= other_c < 8):
        return False

    current_color = get_gem_color(r, c, im)
    other_color = get_gem_color(other_r, other_c, im)

    def color_after_swap(row, col):
        if (row, col) == (r, c):
            return other_color
        if (row, col) == (other_r, other_c):
            return current_color
        return get_gem_color(row, col, im)

    def has_match_at(row, col):
        color = color_after_swap(row, col)
        for dr, dc in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                next_row = row + dr * direction
                next_col = col + dc * direction
                while (0 <= next_row < 8 and 0 <= next_col < 8
                       and color_after_swap(next_row, next_col) == color):
                    count += 1
                    next_row += dr * direction
                    next_col += dc * direction
            if count >= 3:
                return True
        return False

    return has_match_at(r, c) or has_match_at(other_r, other_c)

def can_move_right(r, c, im):
    return _can_swap_and_match(r, c, r, c + 1, im)

def can_move_up(r, c, im):
    return _can_swap_and_match(r, c, r - 1, c, im)

def can_move_down(r, c, im):
    return _can_swap_and_match(r, c, r + 1, c, im)

In [ ]:
import time

start_time = time.time()
im = pyg.screenshot()

while time.time() - start_time < 60:
    for r in range(8):
        if time.time() - start_time >= 60:
            break
        for c in range(8):
            if time.time() - start_time >= 60:
                break

            if can_move_right(r, c, im):
                next_r, next_c = r, c + 1
            elif can_move_up(r, c, im):
                next_r, next_c = r - 1, c
            elif can_move_down(r, c, im):
                next_r, next_c = r + 1, c
            else:
                continue

            x, y = gem_position(r, c)
            next_x, next_y = gem_position(next_r, next_c)
            pyg.moveTo(x, y)
            pyg.dragTo(next_x, next_y, duration=0.2, button='left')
            pyg.sleep(0.1)
            im = pyg.screenshot()